In [1]:
import pandas as pd

# Load both sources
df = pd.read_csv("data/cleaned_crime_long.csv")
df["Date"] = pd.to_datetime(df["Date"])

newer = pd.read_csv("MPS_Borough_Level_Crime _Most_Recent_24_months_new.csv")

# Melt the newer file: wide (one column per month) -> long
month_cols = ["202603", "202604", "202605", "202606"]  # Mar-Jun 2026 only (the new months)

newer_long = newer.melt(
    id_vars=["Group", "SubGroup", "BOCU"],
    value_vars=month_cols,
    var_name="YearMonth",
    value_name="CrimeCount"
)

# Convert "202603" -> proper date (2026-03-01)
newer_long["Date"] = pd.to_datetime(newer_long["YearMonth"], format="%Y%m")
newer_long["Year"] = newer_long["Date"].dt.year
newer_long["Month"] = newer_long["Date"].dt.month

# Rename to match df's column names
newer_long = newer_long.rename(columns={
    "Group": "MajorCategory",
    "SubGroup": "MinorCategory",
    "BOCU": "Borough"
})

# Match column order
newer_long = newer_long[["Borough", "MajorCategory", "MinorCategory", "Date", "Year", "Month", "CrimeCount"]]

# Drop "Unknown" borough to match your cleaning rules
newer_long = newer_long[newer_long["Borough"] != "Unknown"]

print(newer_long.shape)
newer_long.head()

(3956, 7)


,Borough,MajorCategory,MinorCategory,Date,Year,Month,CrimeCount
0,Barking and Dagenham,ARSON AND CRIMINAL DAMAGE,ARSON,2026-03-01,2026,3,3
1,Barking and Dagenham,ARSON AND CRIMINAL DAMAGE,CRIMINAL DAMAGE,2026-03-01,2026,3,116
2,Barking and Dagenham,BURGLARY,BURGLARY BUSINESS AND COMMUNITY,2026-03-01,2026,3,22
3,Barking and Dagenham,BURGLARY,RES BURGLARY OF A HOME,2026-03-01,2026,3,44
4,Barking and Dagenham,BURGLARY,RES BURGLARY OF UNCONNECTED BUILDING,2026-03-01,2026,3,16


In [2]:
# Combine older (through Feb 2026) with newer (Mar-Jun 2026)
combined = pd.concat([df, newer_long], ignore_index=True)

# Sanity checks
print("Combined shape:", combined.shape)
print("Date range:", combined["Date"].min(), "to", combined["Date"].max())
print("Number of boroughs:", combined["Borough"].nunique())
print("Number of major categories:", combined["MajorCategory"].nunique())

Combined shape: (199131, 7)
Date range: 2010-04-01 00:00:00 to 2026-06-01 00:00:00
Number of boroughs: 32
Number of major categories: 13


In [3]:
combined.to_csv("data/cleaned_crime_long_through_june2026.csv", index=False)
print("Saved!")

print(combined["MajorCategory"].unique())

Saved!
['ARSON AND CRIMINAL DAMAGE' 'BURGLARY' 'DRUG OFFENCES'
 'FRAUD AND FORGERY' 'MISCELLANEOUS CRIMES AGAINST SOCIETY'
 'POSSESSION OF WEAPONS' 'PUBLIC ORDER OFFENCES' 'ROBBERY'
 'SEXUAL OFFENCES' 'THEFT' 'VEHICLE OFFENCES'
 'VIOLENCE AGAINST THE PERSON' 'NFIB FRAUD']


In [4]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pandas as pd

exclude_categories = ["FRAUD AND FORGERY", "NFIB FRAUD"]
categories = [c for c in combined["MajorCategory"].unique() if c not in exclude_categories]
boroughs = sorted(combined["Borough"].unique())

results = []
failed = []

for borough in boroughs:
    for category in categories:
        subset = combined[(combined["Borough"] == borough) & (combined["MajorCategory"] == category)]
        monthly = subset.groupby("Date")["CrimeCount"].sum().reset_index().sort_values("Date")
        ts = monthly.set_index("Date")["CrimeCount"]

        # Skip if too little data to fit a seasonal model
        if len(ts) < 24:
            failed.append({"Borough": borough, "Category": category, "Reason": "insufficient data"})
            continue

        try:
            model = SARIMAX(ts, order=(1,1,0), seasonal_order=(1,1,0,12),
                             enforce_stationarity=False, enforce_invertibility=False)
            fit = model.fit(disp=False)
            forecast = fit.forecast(steps=6)  # July-Dec 2026

            results.append({
                "Borough": borough,
                "Category": category,
                "2026-08": round(forecast.iloc[1]),
                "2026-09": round(forecast.iloc[2]),
                "2026-10": round(forecast.iloc[3]),
                "2026-11": round(forecast.iloc[4]),
                "2026-12": round(forecast.iloc[5]),
            })
        except Exception as e:
            failed.append({"Borough": borough, "Category": category, "Reason": str(e)})

    print(f"Finished: {borough}")

borough_category_forecast = pd.DataFrame(results)
print("\nTotal successful:", borough_category_forecast.shape)
print("Total failed:", len(failed))

borough_category_forecast.to_csv("data/borough_category_forecast_aug_dec_2026.csv", index=False)
print("Saved!")

if failed:
    failed_df = pd.DataFrame(failed)
    failed_df.to_csv("data/borough_category_forecast_FAILED.csv", index=False)
    print("Some combos failed - see borough_category_forecast_FAILED.csv")

/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_date

Finished: Barking and Dagenham


/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_date

Finished: Barnet


/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_date

...

/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_date

Finished: Wandsworth


/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_date

Finished: Westminster

Total successful: (352, 7)
Total failed: 0
Saved!


/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


In [5]:
print(borough_category_forecast.shape)
print(borough_category_forecast.head(10))
print("\nFailed:", len(failed))
if failed:
    print(pd.DataFrame(failed))

(352, 7)
                Borough                              Category  2026-08  \
0  Barking and Dagenham             ARSON AND CRIMINAL DAMAGE      134   
1  Barking and Dagenham                              BURGLARY       67   
2  Barking and Dagenham                         DRUG OFFENCES       99   
3  Barking and Dagenham  MISCELLANEOUS CRIMES AGAINST SOCIETY       36   
4  Barking and Dagenham                 POSSESSION OF WEAPONS       15   
5  Barking and Dagenham                 PUBLIC ORDER OFFENCES      127   
6  Barking and Dagenham                               ROBBERY       66   
7  Barking and Dagenham                       SEXUAL OFFENCES       96   
8  Barking and Dagenham                                 THEFT      258   
9  Barking and Dagenham                      VEHICLE OFFENCES      159   

   2026-09  2026-10  2026-11  2026-12  
0      104      112      115      120  
1       72       79       91      101  
2      117      110      130      104  
3       36      